# 트랜스포머

### RNN의 한계
1. RNN: 앞뒤 내용이 서로 연관된 데이터에 대해 기억을 통해 이전 단계에서 처리한 정보를 현재 단계의 입력에 함께 활용하는 신경망 | 모든 스텝에서 동일한 가중치를 반복해서 사용
    * 문제점
        * 장기 의존성 문제 -> 데이터가 길어질 수록, 초반에 위치한 정보가 잘 반영이 되지 않는 문제
        * Vanishing grandient -> 계산 값이 0이 된다
2. LSTM : Vanishing grandient을 해결하고자 중요한 것을 오래 기억하고, 필요 없는 것은 잃어버리자! (Input gate와 Forget gate가 있기 때문에 사용할 아웃풋이 RNN에 비해 적음)
    * 문제점
        * 3가지 게이트를 운용하기 때문에, 계산이 복잡하고 시간이 오래 걸림
        * 여전히 긴 문장에 있어서는 Vanishing gradient가 발생
            * 왜? 초반부 데이터들이 forget gate를 통과하는 횟수가 더 많기 때문

### 어텐션의 등장
1. Seq2Seq : input 내용을 다른 시퀀스로 변환하는 모델 구조
    * 장점
        * 입력과 출력의 단어 개수가 같지 않아도 됨.
        * 음성 변환, 문장 요약 등 형태 및 내용에 있어 변환이 필요한 경우에도 적용 가능
    * 인코더: 중요한 내용들을 Context vector로 표현
    * 디코더: 결과 출력 

2. 어텐션의 등장 배경: seq2seq의 병목 현상
    * 기존 Seq2seq 모델은 인코더가 모든 정보를 압축해서 고정된 크기의 Context Vector 하나로 만드는 구조
    * 입력 문장이 길어지면 -> 압축하는데 정보가 많이 빠짐(데이터소실) 문제!!
    * 따라서, 디코더가 단어를 생성할 때마다 입력 문장 전체를 다시 확인하고 가장 관련 있는 정보들만을 이용해 새로운 context vector를 생성함
    * **결과를 만들 때, 입력값의 모든 부분을 동일하게 보지 말고, 지금! 이순간! 가장 중요한 부분에 더 높은 가중치를 부여해서 참고하자! 라는 핵심 아이디어에서 시작**
    * **출력의 시점마다 새로운 context vector를 동적으로 계산; 병렬화**

###  Attention
* 말 그대로 중요한 정보에 더 집중하는 메커니즘
* 결과를 만들 때 입력 값의 모든 부분을 똑같이 참고하는 것이 아니라, 현재 시점에서 가장 중요한 부분에 더 높은 가중치를 부여하여 활용함
* 즉, 출력 단어를 생성할 때 어떤 단어에 얼마나 집중할지 결정하는 기술
\
Seq2Seq2 와 비교했을 때, 가장 중요한 차이점이 출력 시점마다 새로운 context vextor를 동적으로 계산한다는 것이 장점이다!

### Attention All you Need <br>
* 순차적 계산 구조를 아예 제거하고, 인코더부터 디코더까지 오직 오텐션만을 사용하는 모델은 어떨까? 라는 생각에서 시작
* Attention is All You Need 에서 RNN의 순환 구조를 완전히 제거하고, self-attention에 기반한 Transformer 모델을 제안
* 기존 RNN 기반 모델들을 어텐션만으로 뛰어넘을 수 있게 되며, 거의 모든 데이터를 다룰 수 있게 됨.
* 파급효과
    * LLM의 탄생: BEBT + GPT 의 직접적인 기반이 됨
    * NLP 평정
    * 타분야로의 확장: 컴퓨터 비전, 음성 처리 등 모든 영역으로 확장 시킴

### 트랜스포머란?
: 어텐션만으로 문장의 의미와 구조를 파악하는 모델
* 핵심 기술
    1. Self-Attention :문장 안에서 어떤 단어가 다른 단어들과 얼마나 중요한 관계를 맺고 있는지 한 번에 파악하는 기술. 
    2. Multi-Head Attention : 셀프 어텐션을 여러 개의 머리(Head)로 동시에, **서로 다른 관점에서 실행**
    3. Positional Encoding : 단어의 위치 정보를 벡터에 추가하여 각 단어가 문장의 몇 번째 **위치**에 존재하는지 알 수 있다
* 기존에는 두 RNN 기반 인코더, 디코더 사이를 어텐션으로 연결하였다면, 트랜스포머에서는 인코더, 디코더마저 어텐션으로 해결

## Self-Attention
### 2-1. Query, Key, Value
1. 쿼리: 모델이 수많은 정보 안에서 **가장 관련 있는 정보를 효과적으로 찾아내기** 위해 3가지 역할로 분할!
    *  Query: 질문, 요청 | 지금 당장 내가 알고 싶거나, 초점을 맞추고 있는 대상
    *  Key: 모든 정보들이 달고있는 이름표 또는 색인, 쿼리는 이 키들과 스스로를 비교하며 얼마나 관련이 있는지를 확인한다
    *  Value : 벨류는 키와 한쌍으로 묶여 있는 '실제 내용물'. 관련도 계산이 끝나고 가장 관련성이 높다고 판단된 키가 선택되면, 모델은 그 키에 해당하는 밸류를 가져와 사용
        * K와 Q가 관련이 깊다고 판단되면, V를 가져와 의미를 보강하는데 사용

### 기존 Cross-Attention Vs Self-Attention
1. 기존 Attention on Seq2Seq (크로스 어텐션)
    * 크로스 어텐션은 하나의 시퀀스가 다른 시퀀스를 참고하여 돌아감 -> 디코더가 인코더를 참고하는 과정
    * 두 개의 서로 다른 정보 소스를 연결하는 다리 역할을 함
    * 핵심 과정
        1. 쿼리 만들기
        2. k,v 불러오기
        3. 유사도 계산 -> 전체가 1이 되도록 가중치로 변환
        4. Weighted Sum 
        5. 디코더가 다음 단어를 예측
2. 셀프 어텐션 
    * 문장 자체를 뜯어보며, 내부의 관계를 파악
    * 문장 스스로가 문맥을 이해하는 과정
    * 핵심 과정
        1. 단어의 프로필 만들기(벡터임베딩)
        2. 단어 간 관계 점수 계산 (내정 및 정렬 점수)
            * 관계가 깊을 수록 점수가 높게 나옴
        3. 중요도 배분 (소프트맥스, 어텐션 가중치)
            * 계산된 관계 점수를 총합이 1이 되는 확률값(어텐션 weight)로 변환
            * 특정 단어를 이해하는데 다른 단어들이 각각 몇 %씩 중요한가?를 나타냄
        4. 문맥이 풍부해진 새로운 프로필 생성
            * 문장 내 다른 단어들의 정보를 자신에게 맞게 조합하여, 문맥이 완벽하게 반여된 새로운 벡터(프로필)로 거듭남

###  Scaled Dot-Product Attention
1. Dot-Product Attention
: 어텐션 스코어를 계산하는 가장 기본적인 방법 중 하나
* 기본적인 과정
    1. Attention Score = Q · K
    2. softmax 함수(단어의 중요도)를 적용해서 총합이 1인 attention weight를 구함
    3. 가중치를 v 벡터에 곱하여 최종 결과값을 계산
* 문제점
    * 벡터의 차원이 커질수록 내적의 결과가 너무 커지거나 작아지게 되는 문제가 존재 = Vanishing Gradient

2. Scaled Dot-Product Attention
: 트랜스포머 방식의 표준, 즉 **트랜스포머의 "어텐션 계산 공식" 자체를 의미**
* 그렇다면 기존의 문제를 어떻게 해결 했을까?
: 내적을 진행한 후, 값의 크기를 맞추는 스케일링 과정을 추가
    * 한 번의 나눗셈으로 내적 값이 과도하게 증가하는 것을 방지 -> 소프트맥스 함수가 안정적으로 학습을 진행할 수 있도록 함

* 계산 결과로 얻어지는 벡터
    * 크로스 어텐션: 어떤 단어에 집중해야 하는지를 반영한 Context Vector
    * self-attention: 문장 내 다른 단어들과의 관계를 반영한 문맥적 표현

## Multi-Head Attention 
: 인코딩된 토큰들을 아키텍쳐가 '이해할 때' 사용된다

### 왜 멀티헤드 어텐션을 쓸까? <br>
1. Single-Head 
 * 문장 내 단어들 간의 관계를 파악하기 위해 하나의 가중치 행렬만을 학습하고 번역에 사용
 * 작동방식: 512차원의 입력 벡터 -> 어텐션 가중치 분포 계산 -> 가중치합 -> 새로운 512 차원의 벡터 출력
* 한계: 문법, 의미, 위치 관계 등 모든 종류의 정보를 한 번에 담아 가중 평균을 하기 때문에, 각 단어 간의 중요한 관계를 놓칠 수 있음
2. Multi-Head 
* **한 단어와 다른 단어 간의 관계를 여러 차원으로 나누어 병렬로 학습**
* 작동방식: 512차원의 입력 벡터를 64차원씩 8개의 벡터로 나누고, Attention 스코어를 병렬로 여러 번 계산

### Multi-Head Attention 작동 방식
1. 분할 : 하나의 512 차원의 벡터가 들어옴 -> 이 벡터를 8개의 작은 벡터 그룹(Q,K,V)으로 투영(projection)하여 8개의 서로 다른 관점으로 나눔 (8개의 서로 다른 가중치 행렬을 곱함) <br>
-> 512차원의 큰 문제 하나가 독립적인 64차원짜리 작은 벡터들(각각 8개의 관점 포함)로 분할되었음
2. 병렬 어텐션 계산 - 각 헤드들이 각자 분석 수행
    1. 점수 계산 - 벡터를 내적하여 관련성 점수를 계산
    2. 크기 조절 - 계산된 점수들은 각 헤드 차원의 제곱근으로 나누줌
    3. 가중치 변환(소프트맥스) - 조절된 점수에 소프트 맥스 함수를 적용해서 어텐션 가중치를 만듦.
    4. 가중합: 가중치를 모든 단어의 V 벡터에 곱한뒤 모두 더함
        * 예) Attention_Output_1 = Attention_Weights_1 × (Value_I_1, Value_love_1, Value_you_1 <br>
-> 헤드 1의 관점에서 문맥을 이해한 64차원의 결과 벡터 나옴 -> 헤드 8까지 반복
3. 결합 및 최종 투영 - 8개의 헤드가 각자 내놓은 8개의 분석결과를 합쳐야 함
    * 결합: 8개의 64차원 결과 벡터들을 순서대로 이어붙여 하나의 거대한 512차원 벡터를 만듦
    * 최종 투영: just 나열된 상태이므로, 정보들을 잘 융합하고 최종적으로 정리하기 위해 또 다른 가중치 행렬을 곱해줌 => **최종 출력 벡터**


## 트랜스포머 전체 아키텍쳐

### 1. 트랜스 포머 전처리 단계
* **텍스트를 입력 받고, 그 텍스트를 토큰화하고, 토큰들을 임베딩하고, 포지션 인코딩으로 순서 정보를 추가하여 인코더 레이어에서 실질적인 연산을 수행**

1. 토큰화 : 입력 텍스트를 모델이 처리할 수 있는 단위(토큰)로 나누는 첫번째 단계
    * 토큰화 방법
    1. 각 토큰은 하나의 단어
    2. 문장부호도 별도의 토큰
    3. 일부 단어는 하나 이상의 토큰으로 나뉠 수 있음
2. 임베딩: 토큰을 숫자의 벡터로 변환하는 과정
    * 임베딩의 구조: 임베딩이 많은 숫자로 이루어진 긴 벡터로 단어를 나타냄. 
        * 이때 벡터의 길이 = 벡터의 차원
    * 가장 중요한 것은 단어의 의미를 공간적으로 표현하는 것
        * **유사한 단어는 유사한 숫자(좌표)** 
* 참고: 워드투벡
    1. 신경망 훈련: 신경망에 문장 입력 -> 다음에 올 단어 예측하도록 훈련
    2. 속성 포착: 각 층은 점점 더 깊은 속성을 이해하려고 
    3. 벡터 추출: 예측을 수행하는 신경망의 마지막에서 두번째 레이어에서 나오는 숫자들을 해당 단어의 임베딩으로 사용
    4. 심층 속성
3. 위치 인코딩 : 순서 정보를 임베딩 벡터에 추가하는 단계
    * 왜 필요해용? -> 모든 단어를 병렬적으로 처리해둬서, 단어의 순서를 알지 못함
    * 어떻게 해용? -> 일관된 순서를 따르는 다른 숫자를 추가하여 순서 정보를 기록함

### 트랜스 포머 인코더와 디코더
1. 트랜스포머의 인코더는 입력 문장을 이해하고 요약된 의미벡터로 변환하는 역할을 함
    * 전처리 과정 및 인코더 레이어로 구성되어 있음
        * 인코더 레이어는 뭐지?
            * Multi-Head Attention 
                * 어떤 단어를 집중해서 볼 지 정하기
            * Feed-Forward : 입력 벡터의 차원을 확장하고 비선형 변환을 적용하여 새로운 표현을 생성하는 신경망 구조 (MLP가 2층으로 완전 연결된 구조)
                * 어텐션에서 정한 단어에 대한 심층 분석
    * RNN/LSTM 은 단어를 순서대로 입력하면 트랜스포머는 단어를 한 번에 입력
2.트랜스포머의 디코더는 인코더가 분석한 입력 문장의 의미 벡터를 받아서 출력 문장을 순차적으로 생성하는 역할을 담당
    * 전처리 과정 및 디코더 레이어, 선형레이어, 소프트맥스 레이어로 이루어져 있음
        * 디코더 레이어는 뭐지?
            * Masked Multi-Head Attention
                * 미래 시점의 단어 정보를 참고하지 못하도록 마스크를 적용하는 어텐션 메커니즘
                * 디코더는 왼쪽에서 오른쪽으로 생성하며, 모델이 이미 생성된 단어들만을 기반으로 다음 단어를 예측하도록 강제함
                * Look-Ahead-Mask를 사용하여 현재 단어보다 미래에 있는 토큰의 어텐션 값을 0으로 만들어 무시
            * Encoder-Decoder Multi-Head Attention (Cross-Attention)
                * 디코더가 인코더의 출력을 참고하면서 현재 생성 중인 단어를 입력 문장의 의미와 연결하는 과정 -> 디코더가 입력 문장 중 어떤 부분에 주목해야 하는 지를 학습함

### 전체 모델 및 데이터 흐름

1. 입력 준비 : 문장을 [의미+위치] 벡터로 바꾸기 > 이게 입력이 됨
    * 임베딩 (의미를 담은 숫자 벡터로 바꾸기)
    * 포지셔널 임베딩 (순서를 모르고 위치에 해당하는 벡터를 의미 벡터에 더하기)
2. 인코더 : 입력 문장의 [문맥적 의미] 깊이 이해하기
    * 셀프 어텐션 (의미 파악)
    * 관계 점수 계산 (내적 > softmax 확률 분포로 변환)
    * 가중합 (가중치, 즉 소프트맥스 확률 분포에 벡터를 곱하고 다 더해서 NEW 벡터 생성)
    * 멀티헤드 (여러 개의 어텐션 헤드를 통해 다양한 관점에서 문맥 파악)
3. 디코더 : 번역 문장을 [한 단어씩] 생성하기